## Pydantic

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025DC7530CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025DC75316D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The titlel of the movie")
    year: int = Field(description="The year the movie was releases")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025DC7530CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025DC75316D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titlel of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was releases', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating

In [6]:
model.invoke("Provide the detail about the movie Inception")

AIMessage(content='**Inception** is a 2010 science fiction action film written, directed, and produced by Christopher Nolan. The film is a complex, mind-bending thriller that delves into the concept of shared dreaming and the blurring of reality.\n\n**Plot:**\n\nThe movie follows Cobb (played by Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (played by Ken Watanabe) to perform a task known as "inception" – planting an idea in someone\'s mind instead of stealing one. Saito wants Cobb to convince Robert Fischer (played by Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, including:\n\n1. Arthur (played by Joseph Gordon-Levitt), a point man and logistics 

In [7]:
model_with_structure.invoke("Provide the detail about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

### Message output alongside Parsed stucture

In [12]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure.invoke("Provide the detail about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vj1ysyzcg', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 281, 'total_tokens': 313, 'completion_time': 0.038181768, 'completion_tokens_details': None, 'prompt_time': 0.018185533, 'prompt_tokens_details': None, 'queue_time': 0.052952307, 'total_time': 0.056367301}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1878-95c7-7191-97e9-72234acf4549-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'vj1ysyzcg', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 32, 't

### Nested Stucture

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float|None=Field(None, description="Budget in million USD")
    

In [13]:
model_with_structure = model.with_structured_output(MovieDetail)
model_with_structure.invoke("Provide the detail about the movie Inception")

MovieDetail(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Browning'), Actor(name='Pete Postlethwaite', role='Mello')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

# TypedDict

In [14]:
from typing_extensions import TypedDict, Annotated

In [15]:
class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]
    

In [16]:
model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

# Data Class

In [18]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='512e5e91-5fec-4bac-acbd-0a1a605b2fc2'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6pa2vjrpx', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 288, 'total_tokens': 319, 'completion_time': 0.099059424, 'completion_tokens_details': None, 'prompt_time': 0.044176162, 'prompt_tokens_details': None, 'queue_time': 0.060134311, 'total_time': 0.143235586}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1886-6fce-73e2-9dce-49430fbf8d88-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')